In [32]:
import timeit
import argparse
import torch
import torch.nn as nn
from torch_geometric.loader import TemporalDataLoader
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from tgb.nodeproppred.evaluate import Evaluator

import os, sys
sys.path.insert(0, os.path.abspath('..'))
# This is required here by wandb sweeps.
# sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

# Import the models

import models.mamba_sheaf_diffusion as msd


# Temporal Graph Benchmark: data examples

In [34]:
DATA = "tgbn-trade"
SEED = 42
BATCH_SIZE = 64
# LR = args.lr
# BATCH_SIZE = args.bs
# K_VALUE = args.k_value  
# NUM_EPOCH = args.num_epoch
# SEED = args.seed
# MEM_DIM = args.mem_dim
# TIME_DIM = args.time_dim
# EMB_DIM = args.emb_dim
# TOLERANCE = args.tolerance
# PATIENCE = args.patience
# NUM_RUNS = args.num_run
# NUM_NEIGHBORS = 10


# setting random seed
torch.manual_seed(SEED)

evaluator = Evaluator(name=DATA)
# ==========

# set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# https://chatgpt.com/c/686c55b6-7628-8008-b3fc-693c91aa98c2
# data loading
dataset = PyGNodePropPredDataset(name=DATA, root="datasets")
train_mask = dataset.train_mask
val_mask = dataset.val_mask
test_mask = dataset.test_mask
data = dataset.get_TemporalData()
data = data.to(device)
metric = dataset.eval_metric

train_data = data[train_mask]
val_data = data[val_mask]
test_data = data[test_mask]
num_classes = dataset.num_classes


train_loader = TemporalDataLoader(train_data, batch_size=BATCH_SIZE)
val_loader = TemporalDataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = TemporalDataLoader(test_data, batch_size=BATCH_SIZE)

# Ensure to only sample actual destination nodes as negatives.
min_dst_idx, max_dst_idx = int(data.dst.min()), int(data.dst.max())



raw file found, skipping download
Dataset directory is  /home/zhuowen/anaconda3/envs/nsd/lib/python3.9/site-packages/tgb/datasets/tgbn_trade
loading processed file


In [35]:
from tgb.nodeproppred.dataset import NodePropPredDataset
ds = NodePropPredDataset(name="tgbn-trade", root="datasets", preprocess=True)
data = ds.full_data

raw file found, skipping download
Dataset directory is  /home/zhuowen/anaconda3/envs/nsd/lib/python3.9/site-packages/tgb/datasets/tgbn_trade
loading processed file


In [38]:
# generate a batch of data from train_loader
for batch in train_loader:
    batch = batch.to(device)
    print(batch)
    # print content of batch
    print("Batch size:", batch.num_nodes)
    print("Batch src nodes:", batch.src)
    print("Batch dst nodes:", batch.dst)
    print("Batch edge index:", batch.edge_index)
    print("Batch time:", batch.t)
    print("Batch message:", batch.msg)
    print("Batch y:", batch.y)      # Not sure what y is but it could be labels
    print("n_id", batch.n_id)  # Node IDs in the batch
    # print available attributes in batch
    break 

TemporalData(src=[64], dst=[64], t=[64], msg=[64, 1], y=[64], n_id=[48])
Batch size: 48
Batch src nodes: tensor([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23,
        23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23,
        23, 23, 23, 23, 23, 23, 23, 23, 23, 23], device='cuda:0')
Batch dst nodes: tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
        19, 20, 21, 22,  0,  1,  2,  3, 24,  4, 25,  5,  6, 26, 27, 28, 29, 30,
        31, 32, 33, 34, 35,  7,  8,  9, 10, 11, 36, 37, 38, 12, 13, 39, 40, 41,
        14, 15, 42, 16, 43, 44, 45, 17, 46, 47], device='cuda:0')
Batch edge index: tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23,
         23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23, 23,
     

In [ ]:
import utils.neighbor_loader as LastNeighborLoader
# neighborhood sampler
neighbor_loader = LastNeighborLoader(, size=8, device=device)


AttributeError: 'dict' object has no attribute 'num_nodes'